# 03b — Platform Attribution (Stage 2)

This notebook runs **Stage 2** of the attribution pipeline: assigning cloud records to specific platform entities.

**Requires:** Classified dataset from [03a — Cloud Classification](03a_cloud_classification.ipynb).

**Attribution phases (cloud records only):**
- **Phase 1 — Direct Entity**: Is the contractor itself a platform vendor (AWS, Microsoft, etc.)?
- **Phase 2 — Description-Based**: Use Stage 1 platform identification (RegEx + LLM synthesis)
- **Phase 3 — Pattern Matching**: Two-tier learning from subcontract flows (high confidence) and description patterns (medium confidence)

**Final synthesis** assigns every record a `final_platform`:
- Non-cloud → contractor name (passes through to contractor-level analysis)
- Attributed cloud → platform name (AWS, Azure, Salesforce, etc.)
- Unattributed cloud → contractor name (preserves contractor-level granularity)

---

In [ ]:
import pandas as pd
import numpy as np
import os, importlib.util

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))

def _import_module(name, filepath):
    spec = importlib.util.spec_from_file_location(name, filepath)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

attr_mod = _import_module('platform_attribution',
    os.path.join(PROJECT_ROOT, 'notebooks', '03_multi-stage_attribution_pipeline', 'platform_attribution.py'))

## 1. Load Classified Dataset

In [ ]:
CLASSIFIED_PATH = os.path.join(PROJECT_ROOT, 'data', '02_processed', '03_classified', 'classified_dataset.csv')
MERGED_PATH = os.path.join(PROJECT_ROOT, 'data', '02_processed', '02_merged', 'merged_dataset.csv')

if os.path.exists(CLASSIFIED_PATH):
    df = pd.read_csv(CLASSIFIED_PATH)
    print(f'Loaded classified dataset: {len(df):,} records, ${df["dollars"].sum()/1e9:.1f}B')
    print(f'Cloud records: {df["is_cloud"].sum():,}')
    if 'classification_source' in df.columns:
        has_llm = df['classification_source'].str.contains('llm', na=False).any()
        print(f'LLM classification: {"Yes" if has_llm else "No (RegEx-only)"}')
else:
    print(f'WARNING: Classified dataset not found at {CLASSIFIED_PATH}')
    print(f'Falling back to merged dataset with RegEx-only classification...')
    cls_mod = _import_module('cloud_classification',
        os.path.join(PROJECT_ROOT, 'notebooks', '03_multi-stage_attribution_pipeline', 'cloud_classification.py'))
    merged_df = pd.read_csv(MERGED_PATH)
    df = cls_mod.run_stage1(merged_df)
    print(f'Classified {len(df):,} records (RegEx-only)')

## 2. Phase 1: Direct Entity Attribution

Check whether the `contractor_name` matches a known platform vendor. This catches:
- Direct prime contracts awarded to AWS, Microsoft, Oracle, etc.
- **Subcontracts flowing to platform vendors** — a key finding of the merged approach

In [ ]:
df = attr_mod.attribute_direct_entities(df)

In [ ]:
# Phase 1 deep dive: subcontracts flowing directly to platform vendors
p1 = df[df['platform_phase1'].notna()]
p1_subs = p1[p1['record_type'] == 'subcontract']
p1_primes = p1[p1['record_type'] != 'subcontract']

print(f'Phase 1 — Direct Entity Attribution:')
print(f'  Platform vendor records (prime):       {len(p1_primes):>5,}  ${p1_primes["dollars"].sum()/1e6:>9.1f}M')
print(f'  Platform vendor records (subcontract): {len(p1_subs):>5,}  ${p1_subs["dollars"].sum()/1e6:>9.1f}M')

if len(p1_subs) > 0:
    print(f'\n  Subcontracts going directly to platform vendors:')
    sub_to_platform = p1_subs.groupby('platform_phase1').agg(
        records=('dollars', 'count'),
        dollars=('dollars', 'sum')
    ).sort_values('dollars', ascending=False)
    for platform, row in sub_to_platform.iterrows():
        print(f'    {platform:20s}: {row["records"]:>4,} subs, ${row["dollars"]/1e6:>8.1f}M')

    # Which prime contractors are subcontracting to platforms?
    print(f'\n  Top prime contractors subcontracting directly to cloud platforms:')
    prime_to_platform = p1_subs.groupby(['prime_contractor', 'platform_phase1']).agg(
        dollars=('dollars', 'sum'),
        n=('dollars', 'count')
    ).sort_values('dollars', ascending=False).head(15)
    for (prime, platform), row in prime_to_platform.iterrows():
        if pd.notna(prime):
            print(f'    {str(prime)[:35]:35s} -> {platform:15s} ${row["dollars"]/1e6:>7.1f}M ({row["n"]:,} subs)')

## 3. Phase 2: Description-Based Attribution

For cloud records not yet attributed, use the platform identified in Stage 1 (synthesis of RegEx + LLM). This is the **key integration point** — when LLM results are available, they flow into attribution through the `platform_stage1` column.

In [ ]:
df = attr_mod.attribute_from_descriptions(df)

In [ ]:
# Current attribution coverage after Phases 1-2
cloud_df = df[df['is_cloud'] == True]
total_cloud = len(cloud_df)

attributed_mask = cloud_df['platform_phase2'].notna()
attributed = cloud_df[attributed_mask]
unattributed = cloud_df[~attributed_mask]

print(f'Cloud records: {total_cloud:,}')
print(f'Attributed to platform (Phases 1+2): {len(attributed):,} ({len(attributed)/total_cloud*100:.1f}%)')
print(f'Unattributed: {len(unattributed):,} ({len(unattributed)/total_cloud*100:.1f}%)')

print(f'\nPlatform breakdown (after Phase 2):')
for plat, count in attributed['platform_phase2'].value_counts().items():
    dollars = attributed.loc[attributed['platform_phase2'] == plat, 'dollars'].sum()
    print(f'  {str(plat):25s}: {count:>5,} records, ${dollars/1e6:>9.1f}M')

## 4. Phase 3: Pattern Matching

For each contractor, examine their attributed records. If a contractor has:
- >= 5 attributed contracts AND
- >= 90% consistency with one platform

Then assign that platform to their remaining unattributed cloud records.

### 4.1 Learning Contractor Patterns

For each contractor, examine their **already-attributed** cloud records (from Phases 1-2). If a contractor has:
- >= 5 attributed records, AND
- >= 90% consistency pointing to the same platform

...then learn that contractor-platform association and apply it to their remaining unattributed cloud records.

**Evidence types within patterns:**
- **Subcontract flows** (strongest signal): When a prime contractor subcontracts directly to AWS/Azure/etc., this is actual money flowing to platforms
- **Description patterns** (supporting signal): When attributed records consistently mention the same platform

Both types are captured by `build_contractor_patterns()`, which examines all Phase 1+2 attributed records regardless of how they were attributed.

In [ ]:
print("\n" + "="*70)
print("PHASE 3: PATTERN MATCHING — Learning from Attributed Records")
print("="*70)

# Build patterns: for each contractor, if ≥90% of their attributed cloud records
# point to the same platform (with ≥5 evidence records), learn that pattern
contractor_patterns = attr_mod.build_contractor_patterns(
    df, 
    min_evidence=5,
    consistency_threshold=0.90
)

print(f"\nPatterns learned: {len(contractor_patterns):,} contractors")

if len(contractor_patterns) > 0:
    # Separate subcontract-based evidence from description-based evidence
    # (Subcontract records have record_type='subcontract' in the attributed set)
    cloud_attributed = df[df['is_cloud'] & df['platform_phase2'].notna()]
    
    sub_evidence_contractors = set()
    desc_evidence_contractors = set()
    
    for _, pat_row in contractor_patterns.iterrows():
        contractor = pat_row['contractor']
        contractor_records = cloud_attributed[cloud_attributed['contractor'] == contractor]
        has_sub_evidence = (contractor_records['record_type'] == 'subcontract').any()
        if has_sub_evidence:
            sub_evidence_contractors.add(contractor)
        else:
            desc_evidence_contractors.add(contractor)
    
    print(f"\nEvidence type breakdown:")
    print(f"  Contractors with subcontract evidence: {len(sub_evidence_contractors):,}")
    print(f"  Contractors with description-only evidence: {len(desc_evidence_contractors):,}")
    
    # Show top patterns
    top_patterns = contractor_patterns.sort_values('evidence_count', ascending=False).head(20)
    print(f"\nTop 20 patterns by evidence count:")
    for idx, (_, row) in enumerate(top_patterns.iterrows(), 1):
        evidence_type = "sub" if row['contractor'] in sub_evidence_contractors else "desc"
        print(f"  {idx:2d}. {str(row['contractor_name'])[:40]:40s} -> {str(row['learned_platform']):15s} "
              f"({int(row['evidence_count']):3d} records, {row['consistency']*100:.0f}% consistent, {evidence_type})")
    
    # Platform distribution
    print(f"\nPlatforms identified via patterns:")
    platform_dist = contractor_patterns.groupby('learned_platform').size().sort_values(ascending=False)
    for platform, count in platform_dist.items():
        print(f"  {str(platform):25s}: {count:>4,} contractors")

In [ ]:
# Apply learned patterns to unattributed cloud records
print("\n" + "="*70)
print("APPLYING PATTERNS TO UNATTRIBUTED CLOUD RECORDS")
print("="*70)

# Before
cloud_before = df[df['is_cloud'] == True]
unattributed_before = cloud_before[cloud_before['platform_phase2'].isna()]
print(f"\nBefore pattern matching:")
print(f"  Total cloud records: {len(cloud_before):,}")
print(f"  Unattributed: {len(unattributed_before):,} (${unattributed_before['dollars'].sum()/1e9:.2f}B)")

df = attr_mod.apply_patterns(df, contractor_patterns)

# After
cloud_after = df[df['is_cloud'] == True]
unattributed_after = cloud_after[cloud_after['platform_phase2'].isna()]
newly_attributed = len(unattributed_before) - len(unattributed_after)
newly_dollars = unattributed_before['dollars'].sum() - unattributed_after['dollars'].sum()

print(f"\nAfter pattern matching:")
print(f"  Newly attributed: {newly_attributed:,} records (${newly_dollars/1e9:.2f}B)")
print(f"  Still unattributed: {len(unattributed_after):,} records (${unattributed_after['dollars'].sum()/1e9:.2f}B)")
print(f"  (Unattributed records will use contractor name in HHI calculation)")

## 5. Final Synthesis

In [ ]:
df = attr_mod.synthesize_final_platform(df)

In [ ]:
# Final attribution summary: platform shares
cloud_final = df[df['is_cloud'] == True].copy()
total_cloud_dollars = cloud_final['dollars'].sum()

known_platforms = {'AWS', 'Azure', 'Google Cloud', 'Salesforce', 'Oracle Cloud',
                   'IBM Cloud', 'ServiceNow', 'Workday', 'Multi-cloud'}
platform_records = cloud_final[cloud_final['final_platform'].isin(known_platforms)]

print(f'Total cloud spending: ${total_cloud_dollars/1e9:.2f}B across {len(cloud_final):,} records')
print(f'Attributed to known platforms: {len(platform_records):,} records '
      f'(${platform_records["dollars"].sum()/1e6:.1f}M, '
      f'{platform_records["dollars"].sum()/total_cloud_dollars*100:.1f}%)')

print(f'\nPlatform shares (known platforms, by dollars):')
platform_spending = platform_records.groupby('final_platform')['dollars'].sum().sort_values(ascending=False)
for platform, dollars in platform_spending.items():
    share = dollars / total_cloud_dollars * 100
    n = len(platform_records[platform_records['final_platform'] == platform])
    bar = '#' * int(share / 2)
    print(f'  {platform:25s} {share:6.2f}%  ${dollars/1e6:>9.1f}M  ({n:>5,} records)  {bar}')


In [ ]:
# Attribution method breakdown
print('Attribution method breakdown (cloud records):')
method_summary = cloud_final.groupby('attribution_method').agg(
    records=('dollars', 'count'),
    dollars=('dollars', 'sum')
)
method_summary['pct'] = (method_summary['dollars'] / total_cloud_dollars * 100).round(1)

preferred_order = [
    'phase1_direct',
    'phase2_description',
    'phase3_pattern',
    'cloud_unattributed_contractor'
]
# Keep any unexpected methods at the end
ordered = [m for m in preferred_order if m in method_summary.index]
tail = [m for m in method_summary.index if m not in ordered]
method_summary = method_summary.loc[ordered + tail]

for method, row in method_summary.iterrows():
    print(f'  {str(method):35s} {row["records"]:>6,} records  ${row["dollars"]/1e6:>9.1f}M  ({row["pct"]:>5.1f}%)')


In [ ]:
## 5.5 Attribution Summary Statistics

print("\n" + "="*70)
print("COMPLETE ATTRIBUTION SUMMARY")
print("="*70)

cloud_df = df[df['is_cloud'] == True]
total_cloud_dollars = cloud_df['dollars'].sum()

# Attribution coverage
attributed_to_platform = cloud_df[cloud_df['final_platform'].isin(known_platforms)]
attributed_to_contractor = cloud_df[~cloud_df['final_platform'].isin(known_platforms)]

print(f"\nTotal cloud spending: ${total_cloud_dollars/1e9:.2f}B ({len(cloud_df):,} records)")
print(f"\nAttribution breakdown:")
print(f"  Attributed to platforms: ${attributed_to_platform['dollars'].sum()/1e9:.2f}B "
      f"({attributed_to_platform['dollars'].sum()/total_cloud_dollars*100:.1f}%) "
      f"- {len(attributed_to_platform):,} records")
print(f"  At contractor level:     ${attributed_to_contractor['dollars'].sum()/1e9:.2f}B "
      f"({attributed_to_contractor['dollars'].sum()/total_cloud_dollars*100:.1f}%) "
      f"- {len(attributed_to_contractor):,} records")

# By phase
print(f"\nAttribution by phase:")
phase_summary = cloud_df.groupby('attribution_method').agg({
    'dollars': ['count', 'sum']
}).round(2)
phase_summary.columns = ['records', 'dollars']
phase_summary['pct'] = (phase_summary['dollars'] / total_cloud_dollars * 100).round(1)
phase_summary = phase_summary.sort_values('dollars', ascending=False)

for method, row in phase_summary.iterrows():
    print(f"  {str(method)[:40]:40s}: {int(row['records']):>6,} records, "
          f"${row['dollars']/1e9:>6.2f}B ({row['pct']:>5.1f}%)")

# Confidence levels
print(f"\nConfidence distribution (attributed cloud only):")
conf_col = None
for c in ['confidence', 'classification_confidence', 'attribution_confidence']:
    if c in attributed_to_platform.columns:
        conf_col = c
        break

if conf_col is None:
    print('  No confidence column available.')
else:
    conf_summary = attributed_to_platform.groupby(conf_col).agg({
        'dollars': ['count', 'sum']
    })
    conf_summary.columns = ['records', 'dollars']
    conf_summary['pct'] = (conf_summary['dollars'] / attributed_to_platform['dollars'].sum() * 100).round(1)

    for conf, row in conf_summary.iterrows():
        print(f"  {str(conf):15s}: {int(row['records']):>6,} records, "
              f"${row['dollars']/1e9:>6.2f}B ({row['pct']:>5.1f}%)")


## 6. Save Attributed Dataset

In [ ]:
OUT_DIR = os.path.join(PROJECT_ROOT, 'data', '02_processed', '03_classified')
os.makedirs(OUT_DIR, exist_ok=True)

out_path = os.path.join(OUT_DIR, 'attributed_dataset.csv')
df.to_csv(out_path, index=False)
print(f'Saved: {out_path}  ({len(df):,} rows)')
print(f'Columns: {list(df.columns)}')

---

**Next:** [04 — Platform Concentration & Final Results](04_platform_hhi.ipynb)